In [4]:
import threading
import time
import random
import pynq.lib.rgbled as rgbled
from pynq.overlays.base import BaseOverlay
base = BaseOverlay("base.bit")

In [2]:
#STARVING = WAITING FOR FORKS (NO BLINKS OR SOLID?)
#NAPPING = DONE (SLOW BLINKS)
#EATING = HAS 2 FORKS (FAST BLINKS!)

In [2]:
led4 = rgbled.RGBLED(4)

def blink(t, d, n):
    for i in range(t):
        if button_press.is_set():
            return
        if n==4: 
            led4.on(0x2) # green
            time.sleep(d)
            led4.off()
            time.sleep(d)

        else:
            base.leds[n].toggle()
            time.sleep(d)
    
def button_check():
    while not button_press.is_set():
        button = btns.read()
        #print(f'reading button: {button}')
        if button != 0: 
            button_press.set()
        time.sleep(0.05)
    
def worker_t(_forks, num):
    #print(f'philosopher {num} starting....')
    
    STARVING = True
    EATING = False
    NAPPING = False
    while not button_press.is_set(): # will run until 
        
        if STARVING: 
            #print(f"\n philosopher {num}, waiting for forks")# print message for waiting for the key
            time.sleep(0.01)
            #blink(20, 0.1, num)# noblink if starving
            # try to acquire locks
            # if aquire move to eating
            # if not acquired wait
            #blink at appropriate speed
            fork_counter = 0
            acquired_forks = []
            for n in range(5):
                if _forks[n].acquire(False)==True:
                    acquired_forks.append(_forks[n])
                    fork_counter+=1
                    if fork_counter == 2: # will this prevent from taking too many spoons?
                        EATING = True
                        STARVING = False
                        #print(f"\n philosopher {num}, forks acquired!")# print message for having the key
                        #break
            
        elif EATING: 
            blink(100, 0.1, num)# blink for a while with a different rate
            #blink at appropriate speed
            #release forks
            for n in  range(2):
       #         print(f'releasing forks')
                acquired_forks[n].release() # release forks
            EATING = False
            NAPPING = True
            time.sleep(0.05)# give enough time to the other thread to grab the key     
            #break ## break out of while loop? 
            #move state to NAPPING
        elif NAPPING: 
            #print(f"\n philosopher {num} is napping!")
            blink(20, 1, num)
            time.sleep(0.5)
            # just blink 
        
threads = []
forks = []

for i in range(5):
    fork = threading.Lock() # creates forks !
    forks.append(fork) 

btns = base.btns_gpio
button_press = threading.Event()
threading.Thread(target=button_check,daemon=True).start() # thread for checking button

for i in range(5):
    #time.sleep(0.03)
    t = threading.Thread(target=worker_t, args=(forks, i))
    threads.append(t)
    t.start()

for t in threads:
    name = t.getName()
    t.join()
    print('{} joined'.format(name))      


/tmp/ipykernel_10668/3733678142.py:89: DeprecationWarning: getName() is deprecated, get the name attribute instead
  name = t.getName()


Thread-6 (worker_t) joined
Thread-7 (worker_t) joined
Thread-8 (worker_t) joined
Thread-9 (worker_t) joined
Thread-10 (worker_t) joined


In [5]:
led4 = rgbled.RGBLED(4)

def blink(t, d, n):
    for i in range(t):
        if button_press.is_set():
            return
        if n==4: 
            led4.on(0x2) # green
            time.sleep(d)
            led4.off()
            time.sleep(d)

        else:
            base.leds[n].toggle()
            time.sleep(d)
    
def button_check():
    while not button_press.is_set():
        button = btns.read()
        #print(f'reading button: {button}')
        if button != 0: 
            button_press.set()
        time.sleep(0.05)
    
def worker_t(_forks, num,nap_time,eat_time):
    #print(f'philosopher {num} starting....')
    
    STARVING = True
    EATING = False
    NAPPING = False
    while not button_press.is_set(): # will run until 
        
        if STARVING: 
           # print(f"\n philosopher {num}, waiting for forks")# print message for waiting for the key
            time.sleep(0.01)
            #blink(20, 0.1, num)# noblink if starving
            # try to acquire locks
            # if aquire move to eating
            # if not acquired wait
            #blink at appropriate speed
            fork_counter = 0
            acquired_forks = []
            for n in range(5):
                if _forks[n].acquire(False)==True:
                    acquired_forks.append(_forks[n])
                    fork_counter+=1
                    if fork_counter == 2: # will this prevent from taking too many spoons?
                        EATING = True
                        STARVING = False
                   #     print(f"\n philosopher {num}, forks acquired!")# print message for having the key
                        #break
            
        elif EATING: 
            blink(eat_time, 0.1, num)# blink for a while with a different rate
            #blink at appropriate speed
            #release forks
            for n in  range(2):
       #         print(f'releasing forks')
                acquired_forks[n].release() # release forks
            EATING = False
            NAPPING = True
            time.sleep(eat_time)# give enough time to the other thread to grab the key     
            #break ## break out of while loop? 
            #move state to NAPPING
        elif NAPPING: 
          #  print(f"\n philosopher {num} is napping!")
            blink(nap_time, 1, num)
            time.sleep(0.5)
            # just blink 
        
threads = []
forks = []

for i in range(5):
    fork = threading.Lock() # creates forks !
    forks.append(fork) 

btns = base.btns_gpio
button_press = threading.Event()
threading.Thread(target=button_check,daemon=True).start() # thread for checking button


nap_time = random.randint(1,10)
eat_time = random.randint(11,14)

for i in range(5):
    #time.sleep(0.03)
    t = threading.Thread(target=worker_t, args=(forks, i,nap_time,eat_time))
    threads.append(t)
    t.start()

for t in threads:
    name = t.getName()
    t.join()
    print('{} joined'.format(name))      


/tmp/ipykernel_10668/2694009083.py:93: DeprecationWarning: getName() is deprecated, get the name attribute instead
  name = t.getName()


Thread-12 (worker_t) joined
Thread-13 (worker_t) joined
Thread-14 (worker_t) joined
Thread-15 (worker_t) joined
Thread-16 (worker_t) joined
